In [21]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn import preprocessing


In [22]:
data = pd.read_csv("Desktop/Projects/AgriYield_AI/datasets/ICRISAT_District_Level_Data.csv")
print("Dataset shape: ",data.shape)
print(" columns: ",data.columns.tolist())

Dataset shape:  (12418, 80)
 columns:  ['Dist Code', 'Year', 'State Code', 'State Name', 'Dist Name', 'RICE AREA (1000 ha)', 'RICE PRODUCTION (1000 tons)', 'RICE YIELD (Kg per ha)', 'WHEAT AREA (1000 ha)', 'WHEAT PRODUCTION (1000 tons)', 'WHEAT YIELD (Kg per ha)', 'KHARIF SORGHUM AREA (1000 ha)', 'KHARIF SORGHUM PRODUCTION (1000 tons)', 'KHARIF SORGHUM YIELD (Kg per ha)', 'RABI SORGHUM AREA (1000 ha)', 'RABI SORGHUM PRODUCTION (1000 tons)', 'RABI SORGHUM YIELD (Kg per ha)', 'SORGHUM AREA (1000 ha)', 'SORGHUM PRODUCTION (1000 tons)', 'SORGHUM YIELD (Kg per ha)', 'PEARL MILLET AREA (1000 ha)', 'PEARL MILLET PRODUCTION (1000 tons)', 'PEARL MILLET YIELD (Kg per ha)', 'MAIZE AREA (1000 ha)', 'MAIZE PRODUCTION (1000 tons)', 'MAIZE YIELD (Kg per ha)', 'FINGER MILLET AREA (1000 ha)', 'FINGER MILLET PRODUCTION (1000 tons)', 'FINGER MILLET YIELD (Kg per ha)', 'BARLEY AREA (1000 ha)', 'BARLEY PRODUCTION (1000 tons)', 'BARLEY YIELD (Kg per ha)', 'CHICKPEA AREA (1000 ha)', 'CHICKPEA PRODUCTION (100

In [23]:
yield_cols = [col for col in data.columns if "YIELD (Kg per ha)" in  col]
print("Yield Columns: ")
for col in yield_cols:
    print(col)

Yield Columns: 
RICE YIELD (Kg per ha)
WHEAT YIELD (Kg per ha)
KHARIF SORGHUM YIELD (Kg per ha)
RABI SORGHUM YIELD (Kg per ha)
SORGHUM YIELD (Kg per ha)
PEARL MILLET YIELD (Kg per ha)
MAIZE YIELD (Kg per ha)
FINGER MILLET YIELD (Kg per ha)
BARLEY YIELD (Kg per ha)
CHICKPEA YIELD (Kg per ha)
PIGEONPEA YIELD (Kg per ha)
MINOR PULSES YIELD (Kg per ha)
GROUNDNUT YIELD (Kg per ha)
SESAMUM YIELD (Kg per ha)
RAPESEED AND MUSTARD YIELD (Kg per ha)
SAFFLOWER YIELD (Kg per ha)
CASTOR YIELD (Kg per ha)
LINSEED YIELD (Kg per ha)
SUNFLOWER YIELD (Kg per ha)
SOYABEAN YIELD (Kg per ha)
OILSEEDS YIELD (Kg per ha)
SUGARCANE YIELD (Kg per ha)
COTTON YIELD (Kg per ha)


In [24]:
# ============================================
# STEP 3: CONVERT WIDE DATA INTO CROP-WISE DATA
# ============================================

records = []

# Find all crop yield columns
yield_columns = [
    col for col in data.columns
    if col.endswith("YIELD (Kg per ha)")
]

print("Yield columns found:", len(yield_columns))

for yield_col in yield_columns:

    
    crop = yield_col.replace(" YIELD (Kg per ha)", "")

    area_col = f"{crop} AREA (1000 ha)"
    production_col = f"{crop} PRODUCTION (1000 tons)"

    
    if area_col in data.columns and production_col in data.columns:

        temp = data[
            [
                "Dist Code",
                "Year",
                "State Code",
                "State Name",
                "Dist Name",
                area_col,
                production_col,
                yield_col
            ]
        ].copy()

        
        temp = temp.rename(columns={
            area_col: "Area",
            production_col: "Production",
            yield_col: "Yield"
        })

        
        temp["Crop"] = crop

        records.append(temp)



crop_df = pd.concat(records, ignore_index=True)


crop_df = crop_df[
    [
        "Dist Code",
        "Year",
        "State Code",
        "State Name",
        "Dist Name",
        "Crop",
        "Area",
        "Production",
        "Yield"
    ]
]

print("\nCrop-wise dataset created successfully!")
print("Shape:", crop_df.shape)

display(crop_df.head(10))

Yield columns found: 23

Crop-wise dataset created successfully!
Shape: (285614, 9)


,Dist Code,Year,State Code,State Name,Dist Name,Crop,Area,Production,Yield
0,1,1978,14,Chhattisgarh,Durg,RICE,612.5,362.2,591.35
1,1,1979,14,Chhattisgarh,Durg,RICE,616.8,330.6,535.99
2,1,1980,14,Chhattisgarh,Durg,RICE,634.9,515.6,812.10
3,1,1981,14,Chhattisgarh,Durg,RICE,630.0,506.9,804.60
4,1,1982,14,Chhattisgarh,Durg,RICE,627.9,513.3,817.49
5,1,1983,14,Chhattisgarh,Durg,RICE,626.7,711.0,1134.51
6,1,1984,14,Chhattisgarh,Durg,RICE,632.2,563.8,891.81
7,1,1985,14,Chhattisgarh,Durg,RICE,630.8,699.8,1109.38
8,1,1986,14,Chhattisgarh,Durg,RICE,643.0,525.0,816.49
9,1,1987,14,Chhattisgarh,Durg,RICE,648.0,523.0,807.10


In [25]:

print("Shape:", crop_df.shape)

print("\nNumber of crops:", crop_df["Crop"].nunique())

print("\nCrops:")
print(crop_df["Crop"].unique())

print("\nRecords per crop:")
print(crop_df["Crop"].value_counts())

print("\nMissing values:")
print(crop_df.isnull().sum())

print("\nData types:")
print(crop_df.dtypes)

Shape: (285614, 9)

Number of crops: 23

Crops:
['RICE' 'WHEAT' 'KHARIF SORGHUM' 'RABI SORGHUM' 'SORGHUM' 'PEARL MILLET'
 'MAIZE' 'FINGER MILLET' 'BARLEY' 'CHICKPEA' 'PIGEONPEA' 'MINOR PULSES'
 'GROUNDNUT' 'SESAMUM' 'RAPESEED AND MUSTARD' 'SAFFLOWER' 'CASTOR'
 'LINSEED' 'SUNFLOWER' 'SOYABEAN' 'OILSEEDS' 'SUGARCANE' 'COTTON']

Records per crop:
Crop
RICE                    12418
WHEAT                   12418
KHARIF SORGHUM          12418
RABI SORGHUM            12418
SORGHUM                 12418
PEARL MILLET            12418
MAIZE                   12418
FINGER MILLET           12418
BARLEY                  12418
CHICKPEA                12418
PIGEONPEA               12418
MINOR PULSES            12418
GROUNDNUT               12418
SESAMUM                 12418
RAPESEED AND MUSTARD    12418
SAFFLOWER               12418
CASTOR                  12418
LINSEED                 12418
SUNFLOWER               12418
SOYABEAN                12418
OILSEEDS                12418
SUGARCANE          

In [26]:

# Sort by district, crop and year
crop_df = crop_df.sort_values(
    by=["Dist Code", "Crop", "Year"]
).reset_index(drop=True)

# Previous year's yield
crop_df["Previous_Year_Yield"] = (
    crop_df.groupby(["Dist Code", "Crop"])["Yield"]
    .shift(1)
)

# Previous year's area
crop_df["Previous_Year_Area"] = (
    crop_df.groupby(["Dist Code", "Crop"])["Area"]
    .shift(1)
)

# Previous year's production
crop_df["Previous_Year_Production"] = (
    crop_df.groupby(["Dist Code", "Crop"])["Production"]
    .shift(1)
)

print("Historical features created successfully!")

display(
    crop_df[
        [
            "Year",
            "State Name",
            "Dist Name",
            "Crop",
            "Area",
            "Yield",
            "Previous_Year_Yield",
            "Previous_Year_Area",
            "Previous_Year_Production"
        ]
    ].head(20)
)

Historical features created successfully!


,Year,State Name,Dist Name,Crop,Area,Yield,Previous_Year_Yield,Previous_Year_Area,Previous_Year_Production
0,1978,Chhattisgarh,Durg,BARLEY,0.1,1000.0,NaN,NaN,NaN
1,1979,Chhattisgarh,Durg,BARLEY,0.2,0.0,1000.0,0.1,0.1
2,1980,Chhattisgarh,Durg,BARLEY,0.2,500.0,0.0,0.2,0.0
3,1981,Chhattisgarh,Durg,BARLEY,0.2,500.0,500.0,0.2,0.1
4,1982,Chhattisgarh,Durg,BARLEY,0.1,1000.0,500.0,0.2,0.1
5,1983,Chhattisgarh,Durg,BARLEY,0.1,0.0,1000.0,0.1,0.1
6,1984,Chhattisgarh,Durg,BARLEY,0.1,1000.0,0.0,0.1,0.0
7,1985,Chhattisgarh,Durg,BARLEY,0.1,1000.0,1000.0,0.1,0.1
8,1986,Chhattisgarh,Durg,BARLEY,0.0,0.0,1000.0,0.1,0.1
9,1987,Chhattisgarh,Durg,BARLEY,0.0,0.0,0.0,0.0,0.0


In [27]:

print("Before removing first-year records:", crop_df.shape)

crop_df = crop_df.dropna(
    subset=["Previous_Year_Yield"]
).copy()

print("After removing first-year records:", crop_df.shape)

print("\nMissing values in historical features:")
print(
    crop_df[
        [
            "Previous_Year_Yield",
            "Previous_Year_Area",
            "Previous_Year_Production"
        ]
    ].isnull().sum()
)

Before removing first-year records: (285614, 12)
After removing first-year records: (278461, 12)

Missing values in historical features:
Previous_Year_Yield         0
Previous_Year_Area          0
Previous_Year_Production    0
dtype: int64


In [28]:

target = "Yield"

features = [
    "Year",
    "State Name",
    "Dist Name",
    "Crop",
    "Area",
    "Previous_Year_Yield",
    "Previous_Year_Area",
    "Previous_Year_Production"
]

X = crop_df[features].copy()
y = crop_df[target].copy()

print("Features:")
print(X.columns.tolist())

print("\nTarget:", target)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Features:
['Year', 'State Name', 'Dist Name', 'Crop', 'Area', 'Previous_Year_Yield', 'Previous_Year_Area', 'Previous_Year_Production']

Target: Yield

X shape: (278461, 8)
y shape: (278461,)


In [29]:

print("Minimum year:", crop_df["Year"].min())
print("Maximum year:", crop_df["Year"].max())

print("\nRecords by year:")
print(crop_df["Year"].value_counts().sort_index())

Minimum year: 1979
Maximum year: 2017

Records by year:
Year
1979    7153
1980    7153
1981    7153
1982    7153
1983    7153
1984    7153
1985    7153
1986    7153
1987    7153
1988    7153
1989    7153
1990    7130
1991    7130
1992    7130
1993    7130
1994    7130
1995    6992
1996    7015
1997    7153
1998    7153
1999    7153
2000    7153
2001    7153
2002    7153
2003    7153
2004    7153
2005    7153
2006    7153
2007    7153
2008    7153
2009    7153
2010    7153
2011    7153
2012    7153
2013    7130
2014    7130
2015    7130
2016    7153
2017    7130
Name: count, dtype: int64


In [30]:
# STEP 9: TIME-BASED TRAIN / TEST SPLIT


# Get unique years in chronological order
years = sorted(crop_df["Year"].unique())

print("Available years:")
print(years)

# Use the last 20% of years for testing
split_index = int(len(years) * 0.80)

train_years = years[:split_index]
test_years = years[split_index:]

print("\nTraining years:")
print(train_years)

print("\nTesting years:")
print(test_years)

# Create train and test datasets
train_df = crop_df[crop_df["Year"].isin(train_years)].copy()
test_df = crop_df[crop_df["Year"].isin(test_years)].copy()

print("\nTraining shape:", train_df.shape)
print("Testing shape:", test_df.shape)

Available years:
[np.int64(1979), np.int64(1980), np.int64(1981), np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]

Training years:
[np.int64(1979), np.int64(1980), np.int64(1981), np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.i

In [31]:
# STEP 10: CREATE TRAINING AND TESTING DATA


X_train = train_df[features].copy()
y_train = train_df[target].copy()

X_test = test_df[features].copy()
y_test = test_df[target].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nX_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (221329, 8)
y_train: (221329,)

X_test: (57132, 8)
y_test: (57132,)


In [32]:

# STEP 11: VERIFY TIME SPLIT


print("Training period:")
print(X_train["Year"].min(), "to", X_train["Year"].max())

print("\nTesting period:")
print(X_test["Year"].min(), "to", X_test["Year"].max())

print("\nLatest training year:", X_train["Year"].max())
print("Earliest testing year:", X_test["Year"].min())

if X_train["Year"].max() < X_test["Year"].min():
    print("\n✅ No temporal overlap - split is correct.")
else:
    print("\n❌ Temporal overlap detected.")

Training period:
1979 to 2009

Testing period:
2010 to 2017

Latest training year: 2009
Earliest testing year: 2010

✅ No temporal overlap - split is correct.


In [33]:

# STEP 12: DEFINE FEATURE TYPES


categorical_features = [
    "State Name",
    "Dist Name",
    "Crop"
]

numerical_features = [
    "Year",
    "Area",
    "Previous_Year_Yield",
    "Previous_Year_Area",
    "Previous_Year_Production"
]

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['State Name', 'Dist Name', 'Crop']

Numerical features:
['Year', 'Area', 'Previous_Year_Yield', 'Previous_Year_Area', 'Previous_Year_Production']


In [34]:
# Numerical preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("✅ Preprocessing pipeline created.")

✅ Preprocessing pipeline created.


In [35]:

# STEP 14: LINEAR REGRESSION


linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

print("Training Linear Regression...")

linear_model.fit(X_train, y_train)

print("✅ Linear Regression trained.")

Training Linear Regression...
✅ Linear Regression trained.


In [35]:
# ============================================
# STEP 15: EVALUATE LINEAR REGRESSION
# ============================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Predictions
y_pred_lr = linear_model.predict(X_test)

# Metrics
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Results")
print("=" * 40)
print(f"MAE  : {mae_lr:.4f}")
print(f"RMSE : {rmse_lr:.4f}")
print(f"R²   : {r2_lr:.4f}")

Linear Regression Results
MAE  : 320.9293
RMSE : 705.6754
R²   : 0.7861


In [40]:
# Models
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror"
    )
}

results = []
trained_models = {}

# Train each model
for name, model in models.items():

    print(f"\nTraining {name}...")

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    predictions = pipeline.predict(X_test)

    # Metrics
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    # Store
    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    trained_models[name] = pipeline

    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")


# Create comparison dataframe
results_df = pd.DataFrame(results)

# Sort by R² (higher is better)
results_df = results_df.sort_values(
    by="R2",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

display(results_df)


Training Linear Regression...
MAE  : 320.9293
RMSE : 705.6754
R²   : 0.7861

Training Decision Tree...
MAE  : 296.2134
RMSE : 719.8306
R²   : 0.7774

Training Random Forest...
MAE  : 236.8043
RMSE : 586.2008
R²   : 0.8524

Training Extra Trees...
MAE  : 268.4679
RMSE : 652.9561
R²   : 0.8169

Training Gradient Boosting...
MAE  : 239.5901
RMSE : 587.5802
R²   : 0.8517

Training XGBoost...
MAE  : 241.8570
RMSE : 582.9844
R²   : 0.8540

MODEL COMPARISON


,Model,MAE,RMSE,R2
0,XGBoost,241.857006,582.984379,0.854022
1,Random Forest,236.804304,586.200848,0.852406
2,Gradient Boosting,239.590093,587.580225,0.851711
3,Extra Trees,268.467859,652.956057,0.816877
4,Linear Regression,320.929337,705.675353,0.786113
5,Decision Tree,296.213372,719.830578,0.777446


In [42]:

best_model_name = results_df.loc[0, "Model"]
best_r2 = results_df.loc[0, "R2"]

print("🏆 Best baseline model:", best_model_name)
print(f"R² score: {best_r2:.4f}")

print("\nFull performance:")
display(results_df)

🏆 Best baseline model: XGBoost
R² score: 0.8540

Full performance:


,Model,MAE,RMSE,R2
0,XGBoost,241.857006,582.984379,0.854022
1,Random Forest,236.804304,586.200848,0.852406
2,Gradient Boosting,239.590093,587.580225,0.851711
3,Extra Trees,268.467859,652.956057,0.816877
4,Linear Regression,320.929337,705.675353,0.786113
5,Decision Tree,296.213372,719.830578,0.777446


In [43]:
# ============================================
# STEP 18: IMPORT GRIDSEARCHCV
# ============================================

from sklearn.model_selection import GridSearchCV

print("✅ GridSearchCV imported successfully.")

✅ GridSearchCV imported successfully.


In [37]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                random_state=42,
                n_jobs=1
            )
        )
    ]
)

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=2
)

In [38]:
rf_param_grid = {
    "model__n_estimators": [100, 150],
    "model__max_depth": [15, 25]
}

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=2
)

print("Starting Random Forest GridSearchCV...")
rf_grid.fit(X_train, y_train)

print("\n✅ Completed!")
print("Best parameters:", rf_grid.best_params_)
print("Best CV R²:", rf_grid.best_score_)

Starting Random Forest GridSearchCV...
Fitting 3 folds for each of 4 candidates, totalling 12 fits


/home/ramakrishna/anaconda3/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/ramakrishna/anaconda3/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/ramakrishna/anaconda3/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pi

[CV] END .......model__max_depth=15, model__n_estimators=100; total time= 8.3min
[CV] END .......model__max_depth=15, model__n_estimators=100; total time= 8.7min
[CV] END .......model__max_depth=15, model__n_estimators=100; total time= 8.9min
[CV] END .......model__max_depth=15, model__n_estimators=150; total time=10.1min
[CV] END .......model__max_depth=15, model__n_estimators=150; total time=10.9min
[CV] END .......model__max_depth=15, model__n_estimators=150; total time=11.2min

✅ Completed!
Best parameters: {'model__max_depth': 15, 'model__n_estimators': 150}
Best CV R²: 0.8780787513725996


[CV] END .......model__max_depth=25, model__n_estimators=100; total time=16.9min
[CV] END .......model__max_depth=25, model__n_estimators=100; total time=17.9min
[CV] END .......model__max_depth=25, model__n_estimators=100; total time=18.0min


In [39]:
# ============================================
# EVALUATE TUNED RANDOM FOREST
# ============================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

# Best tuned RF model
best_rf_model = rf_grid.best_estimator_

# Predict on the existing test set
rf_test_pred = best_rf_model.predict(X_test)

# Metrics
rf_test_mae = mean_absolute_error(
    y_test,
    rf_test_pred
)

rf_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_test_pred
    )
)

rf_test_r2 = r2_score(
    y_test,
    rf_test_pred
)

print("=" * 60)
print("TUNED RANDOM FOREST — TEST RESULTS")
print("=" * 60)

print(f"MAE  : {rf_test_mae:.4f}")
print(f"RMSE : {rf_test_rmse:.4f}")
print(f"R²   : {rf_test_r2:.4f}")
print("=" * 60)

TUNED RANDOM FOREST — TEST RESULTS
MAE  : 232.7987
RMSE : 583.7618
R²   : 0.8536


In [40]:
# ============================================
# XGBOOST GRIDSEARCHCV
# ============================================

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBRegressor(
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

xgb_param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [4, 6],
    "model__learning_rate": [0.05, 0.1]
}

xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=2
)

print("Starting XGBoost GridSearchCV...")
print("Total combinations: 8")
print("Total fits: 24")

xgb_grid.fit(X_train, y_train)

print("\n✅ XGBoost tuning completed!")

print("\nBest Parameters:")
print(xgb_grid.best_params_)

print("\nBest CV R²:")
print(xgb_grid.best_score_)

Starting XGBoost GridSearchCV...
Total combinations: 8
Total fits: 24
Fitting 3 folds for each of 8 candidates, totalling 24 fits


/home/ramakrishna/anaconda3/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/ramakrishna/anaconda3/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/ramakrishna/anaconda3/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pi


✅ XGBoost tuning completed!

Best Parameters:
{'model__learning_rate': 0.05, 'model__max_depth': 4, 'model__n_estimators': 200}

Best CV R²:
0.8754417934410331


In [ ]:
# ============================================
# EVALUATE TUNED XGBOOST ON TEST SET
# ============================================

best_xgb_model = xgb_grid.best_estimator_

xgb_test_pred = best_xgb_model.predict(X_test)

xgb_test_mae = mean_absolute_error(
    y_test,
    xgb_test_pred
)

xgb_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        xgb_test_pred
    )
)

xgb_test_r2 = r2_score(
    y_test,
    xgb_test_pred
)

print("=" * 60)
print("TUNED XGBOOST — TEST RESULTS")
print("=" * 60)

print(f"MAE  : {xgb_test_mae:.4f}")
print(f"RMSE : {xgb_test_rmse:.4f}")
print(f"R²   : {xgb_test_r2:.4f}")
print("=" * 60)

TUNED XGBOOST — TEST RESULTS
MAE  : 237.7194
RMSE : 583.0429
R²   : 0.8540


[CV] END model__learning_rate=0.05, model__max_depth=6, model__n_estimators=300; total time=   6.1s
[CV] END .......model__max_depth=25, model__n_estimators=150; total time=19.2min
[CV] END model__learning_rate=0.05, model__max_depth=4, model__n_estimators=200; total time=   2.1s
[CV] END model__learning_rate=0.1, model__max_depth=4, model__n_estimators=200; total time=   2.8s
[CV] END model__learning_rate=0.1, model__max_depth=4, model__n_estimators=300; total time=   3.6s
[CV] END model__learning_rate=0.05, model__max_depth=6, model__n_estimators=300; total time=   6.3s
[CV] END model__learning_rate=0.05, model__max_depth=6, model__n_estimators=300; total time=   6.4s
[CV] END .......model__max_depth=25, model__n_estimators=150; total time=19.9min
[CV] END model__learning_rate=0.05, model__max_depth=4, model__n_estimators=200; total time=   2.6s
[CV] END model__learning_rate=0.1, model__max_depth=4, model__n_estimators=200; total time=   3.2s
[CV] END model__learning_rate=0.1, model_

In [42]:
# ============================================
# FINAL PRODUCTION XGBOOST MODEL
# ============================================

import joblib
import json
import os

print("🚀 Creating final production model...")

final_xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBRegressor(
                objective="reg:squarederror",
                learning_rate=0.05,
                max_depth=4,
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Training final model...")

final_xgb_pipeline.fit(X_train, y_train)

print("✅ Final model trained!")

🚀 Creating final production model...
Training final model...
✅ Final model trained!


In [43]:
# ============================================
# FINAL TEST EVALUATION
# ============================================

final_test_pred = final_xgb_pipeline.predict(X_test)

final_mae = mean_absolute_error(y_test, final_test_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, final_test_pred))
final_r2 = r2_score(y_test, final_test_pred)

print("=" * 60)
print("🏆 FINAL PRODUCTION MODEL")
print("=" * 60)

print(f"MAE  : {final_mae:.4f}")
print(f"RMSE : {final_rmse:.4f}")
print(f"R²   : {final_r2:.4f}")

print("=" * 60)

🏆 FINAL PRODUCTION MODEL
MAE  : 237.7194
RMSE : 583.0429
R²   : 0.8540


In [47]:
import os
import joblib

model_dir = "/home/ramakrishna/Desktop/Projects/AgriYield_AI/backend/models"

os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(
    model_dir,
    "agri_yield_xgboost.joblib"
)

joblib.dump(
    final_xgb_pipeline,
    model_path
)

print("✅ Model saved successfully!")
print(model_path)

✅ Model saved successfully!
/home/ramakrishna/Desktop/Projects/AgriYield_AI/backend/models/agri_yield_xgboost.joblib


In [48]:
import json

metadata = {
    "model_name": "AgriYield XGBoost",
    "algorithm": "XGBRegressor",

    "parameters": {
        "learning_rate": 0.05,
        "max_depth": 4,
        "n_estimators": 200,
        "random_state": 42
    },

    "metrics": {
        "mae": float(final_mae),
        "rmse": float(final_rmse),
        "r2": float(final_r2)
    },

    "target": target,
    "features": features,

    "training_period": "1979-2009",
    "test_period": "2010-2017"
}

metadata_path = os.path.join(
    model_dir,
    "model_metadata.json"
)

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("✅ Metadata saved successfully!")
print(metadata_path)

✅ Metadata saved successfully!
/home/ramakrishna/Desktop/Projects/AgriYield_AI/backend/models/model_metadata.json


In [51]:
print(X_test.head(5).to_string())

    Year    State Name Dist Name    Crop  Area  Previous_Year_Yield  Previous_Year_Area  Previous_Year_Production
32  2010  Chhattisgarh      Durg  BARLEY  0.09               166.67                0.06                      0.01
33  2011  Chhattisgarh      Durg  BARLEY  0.15               777.78                0.09                      0.07
34  2012  Chhattisgarh      Durg  BARLEY  0.19              1133.33                0.15                      0.17
35  2013  Chhattisgarh      Durg  BARLEY  0.22               578.95                0.19                      0.11
36  2014  Chhattisgarh      Durg  BARLEY  0.14               545.45                0.22                      0.12


In [52]:
print(X_train.head(5).to_string())

   Year    State Name Dist Name    Crop  Area  Previous_Year_Yield  Previous_Year_Area  Previous_Year_Production
1  1979  Chhattisgarh      Durg  BARLEY   0.2               1000.0                 0.1                       0.1
2  1980  Chhattisgarh      Durg  BARLEY   0.2                  0.0                 0.2                       0.0
3  1981  Chhattisgarh      Durg  BARLEY   0.2                500.0                 0.2                       0.1
4  1982  Chhattisgarh      Durg  BARLEY   0.1                500.0                 0.2                       0.1
5  1983  Chhattisgarh      Durg  BARLEY   0.1               1000.0                 0.1                       0.1
